# Chapter 03A - How word embeddings learn

**If the input IDs never change, what exactly does training change?**

Earlier lessons gave words IDs and fetched their rows. Now train one CBOW model
on real text and inspect how its embedding table changes. The model's annotated
code handles allocation and lookup; the lesson stays focused on the representation.

| Core path | Evidence you will see |
|---|---|
| 1 - Why learn profiles? | The space-probe analogy, semantic relatedness, and a numbered training flow |
| 2 - Let text supply questions | Stable IDs and sentence-local context/answer windows |
| 3 - Train one embedding model | One CBOW class, actual row changes, and reserved-story loss |
| 4 - Inspect learned representations | Word neighbours, distinct from context-based prediction scores |
| 5 - Explain the remaining limit | A plain-English example of why averaging loses word order |
| 6 - Read, write, and learn | Step-by-step encoder-decoder training and the retained independent comparison |

Use the chapter kernel from [setup](README.md#before-you-start); no earlier notebook
state is required. Running the text model needs the pinned TinyStories file:
the first run downloads it, and later runs verify the local cache.

**Scope:** embeddings are the subject; CBOW is the training vehicle, not a second
lesson in autograd internals. This small model is not a useful general-language
generator. Broader objectives are marked extensions; Run All still executes them.


**Where learning happens:** section 3.3 trains the main CBOW and displays its
own progress. Section 6.3 trains a separate reader/writer. The larger comparison
in 6.5-6.13 deliberately restarts all three models on its own split.


In [ ]:
# %% Setup - Import the tools for one text-trained embedding model
import math
import sys
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Repeat the starting draw and bound this CPU lesson's parallel work.
SEED = 17
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(1)
# Measure the whole run, including any optional extensions executed with Run All.
notebook_started = time.perf_counter()


## 1 - From a fixed ID to a learned row

Imagine you are building a catalogue of objects discovered by space probes.
Every object gets a unique identifier: `Object-417`, `Object-892`, and so on.
The identifiers distinguish the records, but tell you nothing about the objects' properties.

**Predict:** with only those catalogue labels, could you tell which objects resemble Earth?

#### Sensors turn an identity into a descriptive profile

Equip the probes with sensors. Each object now has a profile containing
measurements such as size, temperature, and atmospheric composition. Every
profile uses the same channel layout: compare size with size, temperature with
temperature, and account for the channels' different units and scales.

Keep the richer celestial picture as your mental model: each catalogue entry
has an identity and a multi-channel signature, not just a name.

![Illustrated celestial objects carry catalogue IDs and numerical signature channels, with telemetry links surrounding a central prediction question.](images/03a-celestial-neighborhood-principle.png)
*Conceptual artwork, not astronomical observations or a computation graph. Its
sensor values are fictional, and its word labels connect to the running example.
Use the object/profile panels for intuition; the aligned table below states the
comparison rule. The central question and links foreshadow prediction, a different job.*

Now you can ask **which objects have profiles similar to Earth's**. Similar
names or nearby catalogue numbers are irrelevant; the measured properties
supply the evidence.

![Catalogue labels identify Earth and two discovered objects; aligned sensor channels provide profiles that can be compared with Earth's readings.](images/03a-object-sensor-signatures.png)

- **Registry ID:** which object's record to fetch.
- **Profile:** `D` values describing that object through a shared channel layout.
- **Catalogue:** `V` profiles arranged as `[V, D]`, one object per row.

The name identifies a record. The profile supplies characteristics to work with.
No missing object or guessing system is needed to explain that distinction.

#### Words can have descriptive profiles too

A token ID likewise tells the model **which word** it is handling. An
**embedding** is that word's numerical profile: values that can capture useful
characteristics of how the word is used.

But words do not come with physical sensors. We choose the number of channels,
not a hand-written meaning for each one. We do not need to label words as
"animal" or "action". The values start random, **written in pencil**; training
can gradually shape them into useful features. A property may be spread across
several channels rather than living in one neatly named coordinate.

| Space exploration | Word representation |
|---|---|
| Registry identifier | Token ID: selects a record, not a degree of meaning |
| Sensor profile for one object | Learned embedding for one word |
| Same sensor layout for every object | Same feature-coordinate layout for every word |
| Catalogue of profiles | Embedding matrix `[V, D]` |
| Find planets with profiles close to Earth's | Find words with related learned profiles |

The analogy explains **what the representation is for**. Physical channels have
defined measurements; embedding channels acquire useful patterns through training.
Random word profiles have not learned those patterns yet.

#### Earth-like planets, semantically related words

Earth is your **query profile**. Compare other planets' profiles with it and
rank the candidates by similarity on the chosen, scaled channels. You are no
longer asking whether two catalogue numbers look alike; you are asking whether
the objects share properties you care about.

Now make `cat` the query. Compare its learned row with other word rows. With
suitable training, nearby profiles can reflect shared meaning or linguistic
roles: `cat` and `dog` can share useful features without becoming the same word.
**That is the connection: searching for Earth-like planets is analogous to
searching for semantically related words in a learned feature space.**

![Cat and dog share the practice answers sat and slept, while bird has flew and sang; IDs and rows remain separate.](images/03a-01-shared-word-features.png)
*These are the controlled practice pairs from Chapter 02B, not measured neighbours
from this notebook. Shared prediction jobs can reward useful features in common.*

The boundary matters. Planets are similar on the channels you chose; word rows
are similar in ways shaped by their training task and data. Relatedness is not
synonymy: even opposites such as `hot` and `cold` can occur in similar contexts.
Section 4 inspects the actual learned neighbours, including imperfect ones.

#### Why is that useful beyond finding neighbours?

You want a model to reuse what it learned, not memorize an unrelated rule for
every exact sequence. Shared numerical features give the same downstream
network something reusable to work with: patterns learned with one word can
help with another word or a new combination of familiar words. This is a route
to **generalization**, not a guarantee of it; reserved text tests whether it happened.

But finding a neighbour of `cat` is not deciding what comes after **the cat**.
Another animal may be semantically close; a verb such as `sat` may be a more
appropriate continuation. Prediction needs the **whole available context**,
not just a search around the last word's embedding.

Learned representations supply reusable features. A separate prediction head
uses the available context to score candidate answers; it does not simply
choose the nearest word embedding. Our CBOW exercise predicts a gap in text.

**Complaint:** we have given every word space for a profile. What teaches the
numbers to become useful rather than random?

#### A separate question: how can a prediction task teach the profiles?

**CBOW (continuous bag of words)** is one training exercise, not the definition
of an embedding. Leave the sky catalogue here and return to text. The actual
question below is **the sleepy ___ sat on**, with `cat` kept separately for checking.

Create one CBOW model, then repeat this training flow:

1. **Prepare** integer IDs for the visible clues and a separate answer ID.
2. **Fetch** their current embedding rows inside the model.
3. **Combine and predict:** average matching feature channels, then let the
   subsequent layers turn that context into vocabulary scores.
4. **Check** the scores against the answer with prediction loss.
5. **Backpropagate** the loss after clearing the previous batch's gradients.
6. **Update** the model parameters, including the embedding table, with the optimizer.

The IDs stay fixed; the stored feature values learn. In the celestial artwork,
`cat` is the checking answer, not an extra clue fed into the prediction.
Comparing a word's profile with other profiles and predicting an answer from
context remain different jobs.

**Complaint:** words have no physical sensors. Let ordinary text supply the
feedback, then inspect whether its learned profiles became useful.


## 2 - Let text supply practice questions

Keep **the sleepy cat sat on the mat** for inspection, not as the whole training
diet. Read the first **1,000 TinyStories stories**, split them into sentences, and
append that probe once as an independent final document. The
[pinned dataset](https://huggingface.co/datasets/roneneldan/TinyStories) contains
synthetic children's stories; its cached file is SHA-verified.

Each word gets a fixed address. The model will create one learned profile per
vocabulary entry; you do not supply hand-written semantic features.


In [ ]:
# %% 2 - Let real text supply practice questions
CHAPTER_DIRECTORY = (
    Path(__file__).resolve().parent if "__file__" in globals() else
    next(candidate for candidate in (
        Path.cwd(), Path.cwd() / "learning" / "03-language-model-foundations" / "03-language-models",
    ) if (candidate / "real_text_embeddings.py").is_file())
)
if str(CHAPTER_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(CHAPTER_DIRECTORY))
from real_text_embeddings import read_story_sentences, read_stories, story_sentences, tokenize

INTRO_STORIES = 1_000
CAT_PROBE = "the sleepy cat sat on the mat"
real_documents = read_story_sentences(INTRO_STORIES)
# One independent document at the END, not copies inserted among the stories.
documents = real_documents + [tokenize(CAT_PROBE)]
print(f"Real source: first {INTRO_STORIES:,} pinned Hugging Face TinyStories stories")
print(f"Documents: {len(documents)}")
print("First tokenized document:", documents[0])
print("Last document (appended exactly once):", documents[-1])
print("Tokenized appended document:", tokenize(CAT_PROBE))


### 2.1 - Give retained words stable addresses

Count words, retain common ones, and assign consecutive IDs. Frequency determines
the ordering; an ID is not a frequency or a degree of meaning. `<unk>` gets zero.
Keep every probe word addressable without adding copies or special sampling
weight to its sentence.


In [ ]:
# %% 2.1 - Count, retain, and encode
items = [word for document in documents for word in document]

counts = Counter(items)
itos = ["<unk>"] + [word for word, _ in counts.most_common(1_199)]
# Rare mat still needs an address to be probed. This adds no examples or weight.
# itos => integer to string
# stoi => string to integer
# we need these lookups to convert between words and their integer IDs
# since model takes in and outputs integer IDs rather than raw strings
itos += [word for word in dict.fromkeys(tokenize(CAT_PROBE)) if word not in itos]
stoi = {word: index for index, word in enumerate(itos)}
vocabulary_size = len(itos)

# convert documents from strings to integer IDs
# i.e., each document is now a list of integer IDs
# corresponding to its words, instead of the raw strings
id_documents = [
    # stoi.get(word, stoi["<unk>"]) looks up the integer ID for each word
    # <unk> is the fallback ID for words not in the vocabulary
    [stoi.get(word, stoi["<unk>"]) for word in document]
    for document in documents
]

print(f"Tokens counted: {len(items)}")
print(f"Vocabulary size: {vocabulary_size}")
print("Cat-probe word counts (not weights):",
      {word: counts[word] for word in dict.fromkeys(tokenize(CAT_PROBE))})


### 2.2 - Cut complete windows without crossing documents

`make_document_windows` applies NumPy's built-in `sliding_window_view` separately
to each sentence, then concatenates the rows. It collects neighbours; it does not
yet choose inputs or answers.

Skip short sentences before calling NumPy. Consecutive starts move one token;
seven tokens and width five yield three complete windows. The sentence-local
views share storage, but final concatenation copies their rows.


In [ ]:
# %% 2.2 - Use NumPy's document-local sliding view
def make_document_windows(id_documents, window_width):
    """Keep complete windows in document order; never bridge document boundaries."""
    if window_width < 1:
        raise ValueError("window_width must be at least one.")

    per_document = [
        np.lib.stride_tricks.sliding_window_view(
            np.asarray(document, dtype=np.int64), window_width,
        )
        for document in id_documents
        if len(document) >= window_width
    ]

    # If no document is long enough, return an empty array.
    if not per_document:
        return np.empty((0, window_width), dtype=np.int64)

    # Concatenate windows from each document without crossing boundaries.
    return np.concatenate(per_document)


CONTEXT_RADIUS = 2
window_width = 2 * CONTEXT_RADIUS + 1
# Reuse the cutter here; each objective below chooses its own inputs and answers.
windows = make_document_windows(id_documents, window_width)

print(f"Window width: {window_width}")
print(f"All windows shape: {windows.shape}")
print(f"Unknown window slots: {(windows == 0).mean():.1%}")
print("Our cat document, sliding exactly as in the opening image:")
# Print the last few windows to inspect the cat document
for row in windows[-3:]:
    print([itos[index] for index in row])

# Convert the windows to PyTorch tensors for model input
window_ids = torch.tensor(windows, dtype=torch.long)

# Extract the windows corresponding to the last document (the cat document)
cat_window_ids = window_ids[-3:]


**Your turn:** predict how many complete windows the seven-word probe makes at
width three. Change only `probe_width` to four; the training windows stay width five.
Printed windows carry the explanation; an optional animation follows the core lesson.


In [ ]:
# %% 2.2 - Your turn: Change only the probe width
probe_width = 3
probe_windows = make_document_windows([id_documents[-1]], probe_width)
for start, row in enumerate(probe_windows):
    print(f"start={start}: {' '.join(itos[token_id] for token_id in row)}")
print(f"{len(probe_windows)} complete {probe_width}-word windows; training width stays {window_width}.")


### 2.3 - Turn each window into a question and answer

For **the sleepy cat sat on**, CBOW gets `[the, sleepy, sat, on]` and is checked
against `cat`. The gap is not a mask token or another model input.

First define a batching helper; it only delivers aligned tensors in shuffled
batches. Then select the four clue positions and the center answer.


In [ ]:
# %% 2.3 - Deliver matching inputs and answers
def make_dataloader(input_ids, target_ids, batch_size=512):
    dataset = TensorDataset(input_ids, target_ids)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )


In [ ]:
# %% 2.3 - Remove the center answer from the clues
# Two [N, 2] slices -> [N, 4]: join left/right clues without the center answer.
# torch.cat joins the left and right context words along the second dimension (dim=1)
cbow_contexts = torch.cat(
    [window_ids[:, :CONTEXT_RADIUS], window_ids[:, CONTEXT_RADIUS + 1:]],
    dim=1,
)
# [N, 5] -> [N]: one center answer per row, in the same row order.
cbow_targets = window_ids[:, CONTEXT_RADIUS]
cbow_dataloader = make_dataloader(cbow_contexts, cbow_targets)
cbow_batch_contexts, cbow_batch_targets = next(iter(cbow_dataloader))

print("CBOW: neighbours -> middle word")
print(
    "Illustrated cat-document pair:",
    [itos[index] for index in cbow_contexts[-3].tolist()],
    "->", itos[cbow_targets[-3]],
)
print("Questions:", len(cbow_contexts))
print("Batch inputs:", tuple(cbow_batch_contexts.shape))
print("Batch answers:", tuple(cbow_batch_targets.shape))



## 3 - Let text teach the embedding table

`nn.Embedding` is a trainable layer, just like `nn.Linear`. Its weight belongs
to `model.parameters()`; the usual loss and optimizer train it with the rest
of CBOW. Its forward operation **selects rows**, not a matrix projection.

```text
four word IDs -> four vectors -> average -> Linear -> ReLU -> word scores
```

**Predict:** after averaging and mixing, how does feedback find the original rows?

The lookup keeps the **row addresses**. The mean's backward rule knows **four
occurrences**, so it sends each one a quarter of the incoming gradient.
Lookup backward adds those contributions at the saved addresses. It does not
guess the words from the output or recognize a layer's class later.

**The values mix; the saved addresses do not disappear.** Watch that round
trip in the animation below. Cross-entropy checks scores against `cat`;
backward computes gradients, and a separate optimizer step changes weights.
Unread input rows have zero lookup gradient, even though its default storage
is a full `[V, D]` tensor.


In [ ]:
# %% 3 - Create one embedding table inside CBOW
from language_model import TrainableLanguageModel

EMBEDDING_WIDTH = 32


class CBOW(TrainableLanguageModel):
    def __init__(self, vocabulary_size, embedding_width):
        super().__init__()
        # Allocate one random trainable [V, D] table, not a new table per input.
        self.embedding = nn.Embedding(vocabulary_size, embedding_width)
        self.hidden = nn.Linear(embedding_width, embedding_width // 2)
        self.activation = nn.ReLU()
        self.output = nn.Linear(embedding_width // 2, vocabulary_size)

    def forward(self, context_ids, target_ids=None):
        # The shared trainer passes targets too; this predictor never reads them.
        # [B, C] IDs select existing rows -> [B, C, D]; the IDs do not change.
        # Autograd sends lookup gradients back only to the rows selected here.
        token_vectors = self.embedding(context_ids)
        # CBOW averages clues within each question, not across the batch.
        pooled_context = token_vectors.mean(dim=1)  # [B, D]
        hidden_features = self.activation(self.hidden(pooled_context))
        # NOTE: This returns logits, i.e., the per word score for each word being the target.
        # The "score" is a logit i.e., any number +ve or -ve with no fixed range
        # we need to map to a probability distribution over the vocabulary,
        # which CrossEntropyLoss does internally when computing the loss.
        return self.output(hidden_features)  # [B, V] vocabulary scores


### 3.1 - Create the model, then record predictions before training

Start with **model setup and a shape check**. Create the random weights and
save the embedding table for comparison later. One forward pass on an ordinary
batch returns `[B, V]` vocabulary scores; it does not calculate loss, run
backward, or update weights. The familiar-question baseline follows in its
own block after the measurement helpers.


In [ ]:
# %% 3.1 - Create the untrained model and check score shapes
torch.manual_seed(SEED)

# initialize the model
model = CBOW(vocabulary_size, EMBEDDING_WIDTH)

# the initial embedding table is random, do a deep clone so we can
# compare it with the trained table later.
embedding_before = model.embedding.weight.detach().clone()

print("Trainable embedding table [V, D]:", tuple(embedding_before.shape))

# run the untrained model on the CBOW contexts
untrained_logits = model(cbow_batch_contexts)

print("Input cbow_contexts:", tuple(cbow_batch_contexts.shape))
print("Vocabulary logits:", tuple(untrained_logits.shape))
assert untrained_logits.shape == (len(cbow_batch_contexts), vocabulary_size)


#### Measurement helpers: inspect scores, do not train

`word_probe` reads one question's scores and reports the expected word's
probability, rank, and the actual top five candidates. `report_probe`
compares those measurements before and after training; it is called later.
These functions are diagnostics, not layers or optimization steps.


In [ ]:
# %% 3.1 - Define prediction diagnostics, not training
def word_probe(logits, expected_word, words=itos):
    """Keep probability, rank, and the actual shortlist; never force the answer."""
    # [V] -> [V]: softmax changes scores to shares without changing the word axis.
    chances = logits.detach().softmax(dim=-1)
    expected_id = words.index(expected_word)
    rank = int((chances > chances[expected_id]).sum()) + 1
    top = chances.topk(5)
    return {
        "probability": chances[expected_id].item(),
        "rank": rank,
        "top5": [(words[index], round(chance, 4))
                 for index, chance in zip(top.indices.tolist(), top.values.tolist())],
    }


def report_probe(label, before, after):
    print(f"{label}: correct-word share {before['probability']:.3%} -> "
          f"{after['probability']:.3%}; rank {before['rank']} -> {after['rank']}")
    print("  Before top 5:", before["top5"])
    print("  After top 5: ", after["top5"])
    probability_change = "increased" if after["probability"] > before["probability"] else "decreased"
    rank_change = "improved" if after["rank"] < before["rank"] else (
        "worsened" if after["rank"] > before["rank"] else "stayed the same")
    print(f"  Probability {probability_change}; rank {rank_change}. "
          + ("The expected word is top-ranked." if after["rank"] == 1 else "Not the first guess."))


#### Before training: inspect the Cat probe

Each five-word window drops its center word, at position 2. That leaves
four input IDs and one separate answer:

| Original window | Visible clues passed to the model | Expected answer |
|---|---|---|
| the sleepy cat sat on | the, sleepy, sat, on | cat |
| sleepy cat sat on the | sleepy, cat, on, the | sat |
| cat sat on the mat | cat, sat, the, mat | on |



In [ ]:
# %% 3.1 - Before training: inspect three familiar questions
# [3, 5] -> [3, 4]: the same missing-center job, now on the familiar cat document.
# Before running the whole training loop, inspect the model's predictions for the familiar cat document.
cat_contexts = cat_window_ids[:, [0, 1, 3, 4]]
cat_contexts_before = cat_contexts.clone()
with torch.inference_mode():
    cat_before_logits = model(cat_contexts)
    cbow_before = {word: word_probe(cat_before_logits[row], word)
                   for row, word in enumerate(("cat", "sat", "on"))}

for row, expected_word in enumerate(("cat", "sat", "on")):
    print("Clues:", [itos[index] for index in cat_contexts[row].tolist()],
          "-> expected:", expected_word)
    print("Before training:", cbow_before[expected_word])


#### One question, one round trip

**Cyan:** values forward. **Orange:** gradients backward. **Green:** weights updated.

Follow **the sleepy ___ sat on**, answer **cat**. The active layer lights up
and enlarges; its actual input and output appear underneath. The row IDs stay
visible until feedback arrives home.

[Open the saved animation](images/03a-embedding-forward-backward.gif).
This is the actual CBOW on a copy, not a second training run on the main model.


In [ ]:
# %% 3.1 - Record one real CBOW step without changing the model being trained
import importlib
import real_text_embeddings

# Running kernels can retain helper definitions from before the local file was edited.
importlib.reload(real_text_embeddings)
from real_text_embeddings import record_embedding_routing

routing_question_index = 0  # CHANGE THIS to 1 or 2 to inspect another real window.
routing_contexts = cat_contexts[routing_question_index:routing_question_index + 1].clone()
routing_targets = cat_window_ids[routing_question_index:routing_question_index + 1, CONTEXT_RADIUS].clone()
routing_trace = record_embedding_routing(model, routing_contexts, routing_targets)
routing_weights = routing_trace["weights"]
routing_gradient = routing_trace["gradient"]
routing_head_gradient = routing_trace["head_gradient"]
routing_selected = set(routing_contexts.flatten().tolist())
routing_gradient_rows = set(routing_gradient.ne(0).any(dim=1).nonzero().flatten().tolist())
routing_changed_rows = set(routing_weights[3].ne(routing_weights[0]).any(dim=1).nonzero().flatten().tolist())
print(f"Recorded cat probe: loss {routing_trace['loss']:.4f}; "
      f"{len(routing_changed_rows)} input rows changed by one fresh-SGD step.")


In [ ]:
# %% 3.1 - Follow values forward and gradients backward in one animation
from IPython.display import HTML, display
from real_text_embeddings import plot_embedding_routing, animate_embedding_routing

routing_figure = plot_embedding_routing(routing_trace, itos)
routing_figure.savefig(CHAPTER_DIRECTORY / "images" / "03a-embedding-forward-backward.png", dpi=140)
plt.close(routing_figure)
routing_animation = animate_embedding_routing(routing_trace, itos)
routing_animation.save(CHAPTER_DIRECTORY / "images" / "03a-embedding-forward-backward.gif", writer="pillow", fps=2)
display(HTML(routing_animation.to_jshtml()))


**Remember:** backward fills gradients; only the final green SGD step changes weights.
The unread `cat` input row stays still; its separate prediction head can learn.

<details>
<summary>Static checkpoint: gradients return to their saved row addresses</summary>

![A single CBOW pipeline highlights lookup backward: the saved cat-probe IDs route four actual occurrence gradients into their original embedding rows; unread rows are zero.](images/03a-embedding-forward-backward.png)

</details>

**Boundary:** this exact unused-row result uses fresh SGD, no decay, and untied
input/output weights. Adam history, weight decay, or tied output weights can
also move rows. Values shown are a subset; the checks use the complete tensors.

**Your turn:** change only `routing_question_index` to `2` and rerun the previous
two cells. Predict which rows will light up before watching the new question.


### 3.2 - Reserve different stories before updating the model

Training on the probe does not test generalization. Reserve the **next 200
stories**, which are absent from the first 1,000 training stories. Encode them
with the **already fixed training vocabulary**, keep sentence-local windows,
and inspect at most 2,048 windows. Unknown words remain `<unk>`.

The evaluation helper sums losses in bounded batches and divides by the actual
number of answers. It never calls backward or an optimizer. Record this loss
before and after the same training run; no additional model reset is needed.


In [ ]:
# %% 3.2 - Reserve later stories and keep their vocabulary fixed
# These stories begin after the last training story, before sentence/window construction.
VALIDATION_STORIES = 200
validation_stories = read_stories(INTRO_STORIES + VALIDATION_STORIES)[INTRO_STORIES:]
assert len(validation_stories) == VALIDATION_STORIES
# Split only the reserved stories; never join their endings into training documents.
validation_documents = [
    sentence for story in validation_stories for sentence in story_sentences(story)
]
# Reuse training IDs, including unknown zero, instead of fitting a validation vocabulary.
validation_id_documents = [
    [stoi.get(word, stoi["<unk>"]) for word in document]
    for document in validation_documents
]
# Bound evaluation work while retaining whole, sentence-local window rows.
validation_windows = torch.tensor(
    make_document_windows(validation_id_documents, window_width)[:2_048], dtype=torch.long,
)
# Apply the same four-clue / center-answer objective as the training data.
validation_contexts = validation_windows[:, [0, 1, 3, 4]]
validation_targets = validation_windows[:, CONTEXT_RADIUS]


def heldout_cbow_loss(candidate):
    "Evaluate every retained answer once; never update the candidate's parameters."
    if len(validation_targets) == 0:
        raise ValueError("Held-out evaluation requires at least one target.")
    candidate.eval()
    total_loss = 0.0
    with torch.inference_mode():
        for start in range(0, len(validation_targets), 512):
            scores = candidate(validation_contexts[start:start + 512])
            total_loss += nn.functional.cross_entropy(
                scores, validation_targets[start:start + 512], reduction="sum",
            ).item()
    return total_loss / len(validation_targets)


print(f"Reserved stories: {INTRO_STORIES + 1} through "
      f"{INTRO_STORIES + VALIDATION_STORIES}; windows: {len(validation_windows)}")
print(f"Reserved unknown slots: {validation_windows.eq(0).float().mean():.1%}")


### 3.3 - Train this CBOW now: watch each epoch

**This is where the main CBOW actually learns**, not at the final comparison.
The animation used a copy; this loop updates the original `model` for ten
epochs with Adam. The model, optimizer and shuffled loader are kept throughout.

Watch four different forms of evidence after every epoch:

| Measurement | What it tells you |
|---|---|
| Practice loss | How well the model fitted that epoch's training questions |
| Reserved-story loss | Whether prediction transferred to separate stories |
| Cat probability and rank | What happened to one familiar answer, not every answer |
| Cat row change | Whether that word's actual stored features moved |

Epoch **0** is before any updates. Each `epochs=1` call then performs the
ordinary batch loop: forward, loss, backward, optimizer step. We do not create
a new model or optimizer between these calls.

To repeat this experiment from epoch 0, rerun the model-creation and baseline
blocks in 3.1 first. Rerunning only this training block starts from whatever
weights are currently in the kernel; it does not reset them automatically.


In [ ]:
# %% 3.3 - Train this CBOW now and watch each epoch
validation_loss_before = heldout_cbow_loss(model)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.008)
cbow_started = time.perf_counter()
losses = []
cbow_learning_history = [{
    "epoch": 0, "train_loss": None, "heldout_loss": validation_loss_before,
    "cat_probability": cbow_before["cat"]["probability"],
    "cat_rank": cbow_before["cat"]["rank"], "cat_row_shift": 0.0,
}]
print("CBOW epoch | practice loss | reserved loss | P(cat) | cat rank | cat row change")
print(f"0 | not trained | {validation_loss_before:.4f} | "
      f"{cbow_before['cat']['probability']:.3%} | {cbow_before['cat']['rank']} | 0")
for epoch in range(1, 11):
    # Keep the SAME model, optimizer state, and shuffled loader across epochs.
    epoch_loss = model.fit_language_model(cbow_dataloader, criterion, optimizer, epochs=1)[0]
    losses.append(epoch_loss)
    with torch.inference_mode():
        epoch_cat = word_probe(model(cat_contexts[:1])[0], "cat")
        cat_row_shift = (
            model.embedding.weight[stoi["cat"]] - embedding_before[stoi["cat"]]
        ).norm().item()
    epoch_heldout = heldout_cbow_loss(model)
    cbow_learning_history.append({
        "epoch": epoch, "train_loss": epoch_loss, "heldout_loss": epoch_heldout,
        "cat_probability": epoch_cat["probability"], "cat_rank": epoch_cat["rank"],
        "cat_row_shift": cat_row_shift,
    })
    print(f"{epoch} | {epoch_loss:.4f} | {epoch_heldout:.4f} | "
          f"{epoch_cat['probability']:.3%} | {epoch_cat['rank']} | {cat_row_shift:.4f}")
cbow_seconds = time.perf_counter() - cbow_started

print(f"Loss: {losses[0]:.4f} -> {losses[-1]:.4f}")
print(f"CBOW: {cbow_seconds:.1f}s on one CPU thread (10 epochs, including progress checks)")
with torch.inference_mode():
    cat_after_logits = model(cat_contexts)
    for row, word in enumerate(("cat", "sat", "on")):
        report_probe(f"CBOW missing {word!r}", cbow_before[word],
                     word_probe(cat_after_logits[row], word))

validation_loss_after = heldout_cbow_loss(model)
print(f"Reserved-story loss: {validation_loss_before:.4f} -> {validation_loss_after:.4f}")
if validation_loss_after < validation_loss_before:
    print("The same trained model also improved on these reserved-story questions.")
else:
    print("Practice improvement did not transfer to this reserved-story check.")

embedding_after = model.embedding.weight.detach()
embedding_row_shifts = (embedding_after - embedding_before).norm(dim=1)
print("Embedding rows changed:", (embedding_row_shifts > 0).sum().item())
for word in ("the", "cat", "sat"):
    index = stoi[word]
    print(f"{word} (fixed ID {index}), first four features: "
          f"{embedding_before[index, :4].tolist()} -> {embedding_after[index, :4].tolist()}")
assert torch.any(embedding_row_shifts > 0)
assert torch.equal(cat_contexts, cat_contexts_before)
print("The learned values changed; the integer input IDs did not.")


**Read the learning, not just the code.** The printed rows trace this same
CBOW from epoch 0 through epoch 10. The full before/after feature values below
the table verify actual embedding changes; the word IDs remain unchanged.

The predictor also learned, so a lower loss is not attributable to embeddings
alone. A particular cat probability or rank need not improve every epoch.
Held-out loss and the actual shortlist keep the claim honest.

Section **6.5 onward** later starts fresh models on a different split to
compare objectives. That is a second experiment, not the first training run.


### 3.4 - A falling curve is evidence about fitting, not every particular answer

Read the probe output: a word's rank can improve while its probability falls.
The appended sentence receives no special training weight, so exact recall is
not guaranteed. The next chart compares practice loss with the random-guess
baseline; the reserved-story result above is the separate transfer check.


The chart now shows this CBOW's own practice and reserved-story curves together.
Practice values average the batches *during* an epoch; reserved values inspect
the updated model *after* it. Only the reserved curve has an untrained epoch-0
point. These are not the later fresh-model comparison curves.


In [ ]:
# %% 3.4 - Compare with the random-guess baseline
random_guess_loss = math.log(vocabulary_size)

fig, axis = plt.subplots(figsize=(8, 4))
axis.plot(range(1, len(losses) + 1), losses, color="#2a6f97", linewidth=2,
          label="Practice: average loss during each epoch")
axis.plot(
    [record["epoch"] for record in cbow_learning_history],
    [record["heldout_loss"] for record in cbow_learning_history],
    color="#007f86", linewidth=2, marker="o",
    label="Reserved stories: evaluate after each epoch",
)
axis.axhline(
    random_guess_loss,
    color="#d97706",
    linestyle="--",
    label=f"random guess log(V) = {random_guess_loss:.2f}",
)
axis.set(title="This CBOW is learning here, before the later comparison",
         xlabel="epoch (0 = before training)", ylabel="cross-entropy")
axis.legend(frameon=False)
axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()
plt.close(fig)

print(f"Random-guess baseline: {random_guess_loss:.4f}")
print(f"Final training loss: {losses[-1]:.4f}")
assert losses[-1] < losses[0]
assert losses[-1] < random_guess_loss


## 4 - Inspect what the learned rows now support

Return to the Earth search. A planet's catalogue number selected its profile;
the profile let you rank related candidates. Here `cat` selects a learned row,
and comparing that row with other rows produces a ranked word neighbourhood.

The stored rows are `model.embedding.weight`, not the prediction scores.
Normalize rows and compare their directions with cosine similarity.
Exclude the query itself and `<unk>`; inspect actual neighbours without declaring
that proximity guarantees synonyms. The held-out check tests prediction, while
these neighbours inspect one consequence for the representation.


In [ ]:
# %% 4 - Inspect the learned table
embedding_weights = model.embedding.weight.detach().cpu()
# [V, 32] -> [V, 1]: keep a singleton feature axis so division normalizes EACH word row.
embedding_norms = embedding_weights.norm(dim=1, keepdim=True).clamp_min(1e-12)
normalized_embeddings = embedding_weights / embedding_norms


def most_similar(word, top_k=4):
    if word not in stoi:
        return f"'{word}' is not in the vocabulary"

    similarities = normalized_embeddings @ normalized_embeddings[stoi[word]]
    similarities = similarities.clone()
    similarities[stoi[word]] = -1
    similarities[stoi["<unk>"]] = -1
    neighbour_ids = similarities.topk(min(top_k, vocabulary_size - 2)).indices
    return [
        (itos[index], round(similarities[index].item(), 3))
        for index in neighbour_ids.tolist()
    ]


for word in ["cat", "dog", "mat", "sat"]:
    print(f"{word:>12} -> {most_similar(word)}")


### 4.1 - Word scores are not embedding neighbours

Nearby rows describe related word usage; they are not automatically the best
answers to a particular gap. The model produces vocabulary scores. Choosing
the highest score gives its best guess; sampling is an optional decoding
strategy for later text-generation lessons.

Use the familiar-word probabilities/ranks and held-out loss from section 3.3
to assess prediction. Use the neighbours above to inspect learned profiles.
Neither diagnostic replaces the other.


## 5 - Same clue words, different order: same average

Compare these two questions:

- **the cat ___ the dog**
- **the dog ___ the cat**

Both contain the same four clue words, including **the** twice. Averaging
their word vectors gives the same result, whichever order they arrive in.

Fill the blank with **chased**: who does the chasing changes. But CBOW has
already combined the clues into the same context vector. It cannot tell
which of these two questions it received, even if it guesses the verb correctly.

With fixed model weights:

```text
same clue words and counts -> same average -> same word scores and probabilities
```

Always choose the highest-scoring word, and you get the same answer.

**The limit is order-blindness, not an absence of all semantics.** CBOW can
learn useful word relationships, but this averaging model cannot distinguish
meanings that depend on the order of otherwise identical clues. More training
cannot reconstruct positions that the average discarded.


### 5.1 - Core checkpoint: what did we learn?

| Question | Evidence or explanation |
|---|---|
| What does the embedding layer create? | The real CBOW owns a trainable `[vocabulary_size, 32]` table |
| What does learning change? | Before/after rows change while the probe IDs stay fixed |
| How does feedback find the right rows after mixing? | Saved IDs route occurrence gradients; mean backward divides by the saved count; repeated IDs add |
| Did text training transfer? | Reserved-story loss and the actual probe output report the result |
| Do neighbours prove prediction quality? | Row neighbours inspect representation; familiar-word probabilities/ranks and held-out loss check predictions |
| What is still missing? | The cat/dog example explains how averaging loses who-did-what order |

**Built and measured:** a model-owned embedding table, real-text training,
one-window gradient routing, actual row changes, reserved-story evaluation,
and word neighbours.
**Explained:** semantic relatedness, feature reuse, allocation and lookup shapes,
the numbered training flow, matching tokenizer/table artifacts, fixed inference
weights, vocabulary-score interpretation, and averaging's order limitation.
**Out of scope for the core:** detailed autograd/registration implementation,
useful general-language quality, contextual Transformer states, and decoding
strategies such as sampling, which belong with text generation.

**The core lesson ends here.** Continue to [03B - Counts and Fixed Windows](03B-counts-and-fixed-window-language-models.ipynb):
would keeping words in separate seats fix the information CBOW discarded?
You do not need the extensions below before taking that next step.


For the ordered reader/writer, continue to **section 6** after the optional A/B
asides. Its training and embedding evidence are separate from the CBOW above;
neither is postponed until the larger comparison.


## After CBOW - Optional views, then the section 6 reader/writer

Sections 1-5 already trained and evaluated the main CBOW. The following
experiments do not continue updating its weights: views read that table,
while other prediction jobs create their own fresh models.

| Route | Additional question |
|---|---|
| A - Optional views and API alternatives | How can we inspect or fuse an already-understood operation? |
| B - Optional Skip-gram | What changes when the center predicts a neighbour instead? |
| 6.1-6.4 - Encoder-decoder training | How do ordered, multi-token answers teach a shared embedding? |
| 6.5-6.13 - Independent larger comparison | What happens when all three objectives train fresh models on a new split? |

Run All still executes every retained experiment. You can also take the
shorter 03B route after the CBOW checkpoint.


## A - Optional views and API alternatives

### A.1 - Optional fused lookup and mean

`EmbeddingBag(mode="mean")` fuses the two operations already used by CBOW.
Copy the **trained core table** for an equality check; no new learning mechanism
or positional awareness is being introduced.


In [ ]:
# %% A.1 - Compare fused lookup with the same trained rows
preview_contexts = cbow_contexts[:3]
# ── Where EmbeddingBag fits ──
embedding_bag = nn.EmbeddingBag(
    vocabulary_size,
    EMBEDDING_WIDTH,
    mode="mean",
)
with torch.no_grad():
    embedding_bag.weight.copy_(model.embedding.weight)

# [3, 4, 32] -> [3, 32]: average each question's clues; EmbeddingBag fuses this same job.
manual_means = model.embedding(preview_contexts).mean(dim=1)
bag_means = embedding_bag(preview_contexts)

print("Manual mean shape:", tuple(manual_means.shape))
print("EmbeddingBag shape:", tuple(bag_means.shape))
print("Maximum difference:", (manual_means - bag_means).abs().max().item())
assert torch.allclose(manual_means, bag_means)


### A.2 - Optional two-dimensional view

Project selected 32-number rows onto two PCA directions for a sketch.
Nearby dots can still differ in omitted directions; keep the full-row cosine
check above as the numerical evidence.


In [ ]:
# %% A.2 - Project learned rows for inspection
# [V, 32] -> [visible words, 32]: keep the running example beside common words.
visible_words = list(dict.fromkeys(tokenize(CAT_PROBE) + [w for w, _ in counts.most_common(10)]))
visible_ids = torch.tensor([stoi[word] for word in visible_words])
visible_vectors = embedding_weights[visible_ids]
# [words, 32] -> [1, 32]: one center per feature, broadcast back across words.
centered = visible_vectors - visible_vectors.mean(dim=0, keepdim=True)
_, _, right_singular_vectors = torch.linalg.svd(centered, full_matrices=False)
# [2, 32] -> [32, 2]: turn two PCA directions into columns; project [words, 32] -> [words, 2].
coordinates = centered @ right_singular_vectors[:2].T

fig, axis = plt.subplots(figsize=(10, 7))
axis.scatter(coordinates[:, 0], coordinates[:, 1], color="#2a6f97", s=55)
for point, token_id in zip(coordinates, visible_ids.tolist()):
    axis.annotate(itos[token_id], point, xytext=(5, 5), textcoords="offset points")
axis.axhline(0, color="#9ca3af", linewidth=0.7)
axis.axvline(0, color="#9ca3af", linewidth=0.7)
axis.set(
    title="Cat-probe and frequent embedding rows projected from 32D to 2D",
    xlabel="projection axis 1",
    ylabel="projection axis 2",
)
axis.grid(alpha=0.15)
plt.tight_layout()
plt.show()
plt.close(fig)


### A.3 - Optional window animation

The printed windows already show the three complete slices. This animation uses
those same rows to highlight the moving window and the hidden center.
Its drawing implementation is optional; it adds no new training mechanism.


In [ ]:
# %% A.3 - Animate the already-built document windows
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation

window_story_words = tokenize(CAT_PROBE)
window_story_frames = [
    [itos[token_id] for token_id in row] for row in cat_window_ids.tolist()
]
assert window_story_frames == [
    window_story_words[start:start + window_width]
    for start in range(len(window_story_words) - window_width + 1)
]


def draw_cbow_window(axis, frame):
    axis.clear()
    axis.set(xlim=(-0.6, len(window_story_words) - 0.4), ylim=(-1.1, 2.0))
    axis.axis("off")
    axis.set_title(
        f"start={frame}: document[{frame}:{frame + window_width}]", fontsize=11,
    )
    for position, word in enumerate(window_story_words):
        active = frame <= position < frame + window_width
        axis.text(
            position, 1.3, word, ha="center", va="center", fontsize=10,
            color="#111827" if active else "#9ca3af",
            bbox=dict(boxstyle="round,pad=0.3", edgecolor="#94a3b8",
                      facecolor="#dbeafe" if active else "#f3f4f6"),
        )
    row = window_story_frames[frame]
    question = row[:CONTEXT_RADIUS] + ["___"] + row[CONTEXT_RADIUS + 1:]
    axis.text(3, 0.6, "Window: " + " ".join(row),
              ha="center", fontsize=10, color="#1e3a8a")
    axis.text(3, -0.05, "CBOW clues: " + " ".join(question),
              ha="center", fontsize=10, color="#007f86")
    axis.text(3, -0.7, "Checked answer: " + row[CONTEXT_RADIUS],
              ha="center", fontsize=11, color="#bf5700", weight="bold")


print(f"Prediction check: {len(window_story_frames)} complete windows, not one per word.")
print("CBOW answers:", [row[CONTEXT_RADIUS] for row in window_story_frames])
print("Watch the blue window move; CBOW keeps each orange answer out of its input.")
window_animation_figure, window_animation_axis = plt.subplots(figsize=(9, 3.2))
window_animation = FuncAnimation(
    window_animation_figure,
    lambda frame: draw_cbow_window(window_animation_axis, frame),
    frames=len(window_story_frames), interval=1300, repeat=False,
)
plt.close(window_animation_figure)
display(HTML(window_animation.to_jshtml(default_mode="once")))


## B - Optional Skip-gram

### B.1 - Skip-gram reverses the prediction question

CBOW uses four neighbours to predict the center. Skip-gram uses the center to
predict **one neighbour per example**. Each five-word window creates four
independent pairs, not one four-word answer:

```text
cat -> the     cat -> sleepy     cat -> sat     cat -> on
```

Keep the same text and vocabulary, but create a new pair dataset and fresh model.
**Predict:** with `N` windows and four neighbours each, how many examples result?


In [ ]:
# %% B.1 - Build center-to-neighbour pairs
# Each raw window supplies one center and all its neighbours.
skipgram_centers = window_ids[:, CONTEXT_RADIUS]
# [N, 2] plus [N, 2] -> [N, 4]: keep each window's neighbours in left-to-right order.
skipgram_neighbours = torch.cat(
    [window_ids[:, :CONTEXT_RADIUS], window_ids[:, CONTEXT_RADIUS + 1:]],
    dim=1,
)
neighbours_per_window = skipgram_neighbours.shape[1]
# [N] -> [4N]: repeat each center four times BEFORE moving to the next window.
skipgram_inputs = skipgram_centers.repeat_interleave(neighbours_per_window)
# [N, 4] -> [4N]: row-major flattening keeps each answer aligned with its repeated center.
skipgram_targets = skipgram_neighbours.reshape(-1)
skipgram_dataloader = make_dataloader(skipgram_inputs, skipgram_targets)
skipgram_batch_inputs, skipgram_batch_targets = next(iter(skipgram_dataloader))

print("Skip-gram: middle word -> ONE neighbour per question")
print("Questions:", len(skipgram_inputs))
print("Batch inputs:", tuple(skipgram_batch_inputs.shape))
print("Batch answers:", tuple(skipgram_batch_targets.shape))
print("Pairs from the illustrated cat window:")
for center, neighbour in zip(
    skipgram_inputs[-3 * neighbours_per_window:-2 * neighbours_per_window],
    skipgram_targets[-3 * neighbours_per_window:-2 * neighbours_per_window],
):
    print("   ", itos[center], "->", itos[neighbour])

assert skipgram_inputs.shape == (len(window_ids) * neighbours_per_window,)
assert skipgram_targets.shape == skipgram_inputs.shape
assert torch.equal(
    # [4N] -> [N, 4]: reconstruct groups to check alignment, not mix examples.
    skipgram_inputs.reshape(len(window_ids), neighbours_per_window),
    # [N] -> [N, 1] -> [N, 4]: broadcast a center beside each of its own neighbours.
    skipgram_centers[:, None].expand_as(skipgram_neighbours),
)
assert torch.equal(
    # [4N] -> [N, 4]: the flattened answers must reconstruct the original neighbour rows.
    skipgram_targets.reshape_as(skipgram_neighbours), skipgram_neighbours,
)
assert len(skipgram_batch_inputs) == len(skipgram_batch_targets)


### B.2 - Reuse the training loop; change the inputs

The forward input is `[B]`, its lookup is `[B, D]`, and its scores are `[B, V]`.
There is no context mean because each example supplies one center.
Train for four epochs and inspect its actual neighbour distributions.
Full-vocabulary scoring keeps the route visible; negative sampling is left for
larger Word2Vec implementations.


In [ ]:
# %% B.2 - Train and inspect Skip-gram
class SkipGram(TrainableLanguageModel):
    def __init__(self, vocabulary_size, embedding_width):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, embedding_width)
        self.hidden = nn.Linear(embedding_width, embedding_width // 2)
        self.activation = nn.ReLU()
        self.output = nn.Linear(embedding_width // 2, vocabulary_size)

    # Same layers as CBOW. The only change is what goes in: one word, not four.
    def forward(self, center_ids, target_ids=None):
        # center_ids: one word per lesson, e.g. cat.                  [B]
        # Look up its row. Only one row, so there's nothing to average. [B, D]
        center_vectors = self.embedding(center_ids)

        # Projection [B, D] -> [B, D/2]: mix this one center's row into features.
        hidden_features = self.activation(self.hidden(center_vectors))

        # Projection [B, D/2] -> [B, V]: score all words as neighbours of this center.
        # The same center appears in several lessons with different answers
        # (cat -> the, sleepy, sat, on), so it can never be sure of just one.
        return self.output(hidden_features)


torch.manual_seed(SEED)
skipgram_model = SkipGram(vocabulary_size, EMBEDDING_WIDTH)
with torch.inference_mode():
    # [1] -> [1, V] -> [V]: one cat-center distribution for the neighbour job.
    skipgram_before_logits = skipgram_model(torch.tensor([stoi["cat"]]))[0]
    skipgram_before = {word: word_probe(skipgram_before_logits, word)
                       for word in ("the", "sleepy", "sat", "on")}
skipgram_optimizer = torch.optim.Adam(skipgram_model.parameters(), lr=0.008)
skipgram_started = time.perf_counter()
skipgram_losses = skipgram_model.fit_language_model(
    skipgram_dataloader,
    criterion,
    skipgram_optimizer,
    epochs=4,
)
skipgram_seconds = time.perf_counter() - skipgram_started

with torch.inference_mode():
    probe_center = skipgram_centers[:1]
    neighbour_logits = skipgram_model(probe_center)
    predicted_ids = neighbour_logits.topk(neighbours_per_window, dim=1).indices[0]

print(f"Skip-gram loss: {skipgram_losses[0]:.4f} -> {skipgram_losses[-1]:.4f}; "
      f"{skipgram_seconds:.1f}s (4 epochs)")
print("Center:", itos[probe_center.item()])
print("Top predicted neighbours:", [itos[index] for index in predicted_ids.tolist()])
with torch.inference_mode():
    cat_neighbour_logits = skipgram_model(torch.tensor([stoi["cat"]]))[0]
    for word, before in skipgram_before.items():
        report_probe(f"Skip-gram cat -> neighbour {word!r}", before,
                     word_probe(cat_neighbour_logits, word))

assert skipgram_losses[-1] < skipgram_losses[0]
assert skipgram_losses[-1] < math.log(vocabulary_size)
assert neighbour_logits.shape == (1, vocabulary_size)


## 6 - Encoder-decoder: read, write, and train the embeddings

### 6.1 - Align the source and answer

This is a different prediction job, **not the smallest fix for CBOW's lost order**.
You can go directly to 03B instead; [03C](03C-fixed-windows-to-recurrent-language-models.ipynb)
and [03D](03D-recurrent-encoder-decoder-to-attention.ipynb) build recurrent reading
and attention more slowly.

Reuse each window as an ordered continuation:

| Role | Example |
|---|---|
| Source | `the sleepy cat` |
| Full target | `<bos> sat on <eos>` |
| Writer input during training | `<bos> sat on` |
| Correct next tokens | `sat on <eos>` |

The encoder GRU passes its final hidden state to a decoder GRU. Training supplies
correct previous target tokens (**teacher forcing**); generation must use its own
guesses. Fixed lengths mean no padding is needed in this preview.


The CBOW in section 3.3 already learned. Here you create a **different, fresh**
model with the same familiar vocabulary plus start/stop markers. Its one
embedding table is shared by the reader and writer, not borrowed from CBOW.


In [ ]:
# %% 6.1 - Build aligned source and target sequences
BOS_ID = vocabulary_size
EOS_ID = vocabulary_size + 1
sequence_itos = itos + ["<bos>", "<eos>"]
sequence_vocabulary_size = len(sequence_itos)

# [N, 5] -> [N, 3] source and [N, 2] continuation: split WITHIN each window.
source_sequences = window_ids[:, :3]
continuations = window_ids[:, 3:]
# [N, 1] + [N, 2] + [N, 1] -> [N, 4]: bracket each answer, never join documents.
sequence_targets = torch.cat(
    [
        torch.full((len(windows), 1), BOS_ID, dtype=torch.long),
        continuations,
        torch.full((len(windows), 1), EOS_ID, dtype=torch.long),
    ],
    dim=1,
)
sequence_dataloader = make_dataloader(source_sequences, sequence_targets)

print("Source shape:", tuple(source_sequences.shape))
print("Target shape:", tuple(sequence_targets.shape))
print("Source:", [sequence_itos[index] for index in source_sequences[-3].tolist()])
print("Target:", [sequence_itos[index] for index in sequence_targets[-3].tolist()])

assert source_sequences.shape == (len(windows), 3)
assert sequence_targets.shape == (len(windows), 4)

sequence_batch_inputs, sequence_batch_targets = next(iter(sequence_dataloader))
print("Sequence batch sources:", tuple(sequence_batch_inputs.shape))
print("Sequence batch full answers:", tuple(sequence_batch_targets.shape))
print("Writer sees:", [sequence_itos[index] for index in sequence_targets[-3, :-1].tolist()])
print("Loss checks:", [sequence_itos[index] for index in sequence_targets[-3, 1:].tolist()])
assert len(sequence_batch_inputs) == len(sequence_batch_targets)
assert torch.equal(source_sequences, window_ids[:, :3])
assert torch.equal(sequence_targets[:, 1:-1], window_ids[:, 3:])
assert bool((sequence_targets[:, 0] == BOS_ID).all())
assert bool((sequence_targets[:, -1] == EOS_ID).all())


### 6.2 - Follow one answer through the reader, writer, and embedding table

Keep the familiar example: read **the sleepy cat**, then practise writing
**sat on <eos>**. `<bos>` starts the writer; `<eos>` is the answer saying stop.

![An ordered observing log feeds a reader's learned briefing, then a writer's word-by-word report; the actual text source is the sleepy cat and the training answer is sat on eos.](images/03a-encoder-decoder-observatory.png)

The stored embedding is a reusable word profile. The GRU's hidden state is
instead a changing note about this particular sequence. Do not confuse them.

1. **Look up the source words.** `the`, `sleepy`, and `cat` select rows from
   the shared embedding table. The encoder reads those vectors in order.
2. **Hand over the reader's final note.** This hidden state initializes the
   decoder; it is not an average of source embeddings.
3. **Start the writer with `<bos>`.** Its embedding and the reader's note let
   the decoder produce vocabulary scores for the first answer, `sat`.
4. **Give the correct previous word during practice.** Feed `sat` to predict
   `on`, then `on` to predict `<eos>`. This is teacher forcing: the current
   answer is checked, not supplied as the input for that same prediction.
5. **Calculate prediction loss at each step.** Cross-entropy checks scores
   against `sat`, `on`, and `<eos>`. The shared trainer combines their losses.
6. **Send feedback backward through both GRUs.** Decoder errors reach the
   writer's input embeddings and, through its initial state, the encoder
   and the source-word embeddings. This is backpropagation through time.
7. **Add feedback into one shared table.** Encoder and decoder lookups use
   the same Parameter. Their occurrence gradients accumulate at the saved
   row IDs; a word used in both paths receives contributions from both.
8. **Apply the optimizer step.** The shared embedding, both GRUs, and the
   vocabulary head learn together. IDs remain fixed. There is no separate
   command saying "train the encoder embeddings".

| Writer input | Correct next word checked by loss |
|---|---|
| `<bos>` | `sat` |
| correct previous `sat` | `on` |
| correct previous `on` | `<eos>` |

**Predict:** can the source-only `cat` row learn even when the writer never
receives `cat` as its previous answer? Yes: feedback reaches it through the
reader's final note. The next block measures shared-table changes.


In [ ]:
# %% 6.2 - Define the reader, writer, and shared embedding
class EncoderDecoderLM(TrainableLanguageModel):
    def __init__(self, vocabulary_size, embedding_width, hidden_width):
        super().__init__()
        # Reader and writer share one registry of input profiles.          [V, D]
        self.embedding = nn.Embedding(vocabulary_size, embedding_width)
        # The reader: walks the source left to right, updating a running note.
        self.encoder = nn.GRU(embedding_width, hidden_width, batch_first=True)
        # The writer: starts from the reader's final note and writes word by word.
        self.decoder = nn.GRU(embedding_width, hidden_width, batch_first=True)
        # Turns the writer's note at each step into a score for every word.
        self.output = nn.Linear(hidden_width, vocabulary_size)

    def encode(self, source_ids):
        # source_ids: e.g. [the, sleepy, cat]                          [B, 3]
        # Look up rows, read them in order. We keep only the final note
        # (the "summary card"), not the note after each word.          [1, B, H]
        # Lookup expands [B, 3] -> [B, 3, D]; GRU keeps its final state [1, B, H].
        _, hidden = self.encoder(self.embedding(source_ids))
        return hidden

    def decode(self, decoder_input_ids, hidden):
        # decoder_input_ids: the words the writer is shown, e.g. [<bos>, sat, on]
        # Starting from the reader's note, write one note per step.    [B, T, H]
        # Lookup [B, T] -> [B, T, D]; recurrence produces [B, T, H] states in order.
        decoder_states, hidden = self.decoder(
            self.embedding(decoder_input_ids),
            hidden,
        )
        # Projection changes [B, T, H] -> [B, T, V], retaining batch and time.
        # Score every word at every step: "what comes next?"
        return self.output(decoder_states), hidden

    def forward(self, source_ids, target_ids=None):
        if target_ids is None:
            raise ValueError("Training requires target IDs for teacher forcing.")
        # 1. Read the source into one summary note.
        hidden = self.encode(source_ids)
        # 2. Teacher forcing: show the writer the CORRECT previous words
        #    (<bos>, sat, on), dropping the last one, which it never needs to see.
        #    targets_for_loss() below supplies the matching answers (sat, on, <eos>).
        # [B, 4] -> [B, 3]: drop the last answer; keep <bos> and previous words.
        logits, _ = self.decode(target_ids[:, :-1], hidden)
        return logits

    def targets_for_loss(self, target_ids):
        # forward() fed the writer target_ids[:, :-1]  -> <bos>, sat, on
        # so its answers are shifted one step left    -> sat,   on,  <eos>
        # Each input word is marked against the word that should come next.
        # [B, 4] -> [B, 3]: drop <bos> to align each score with the NEXT token.
        return target_ids[:, 1:]


### 6.3 - Train this reader/writer now and inspect its learned rows

Create its fresh model once, keep the same optimizer for six epochs,
and print progress from the untrained epoch 0 onward. Alongside practice
loss and the familiar first-word probability, track the shared table's
actual movement from its initial values. Both parts, not just the head,
are being optimized in this run.


In [ ]:
# %% 6.3 - Train the reader/writer and inspect its embedding changes
torch.manual_seed(SEED)
sequence_model = EncoderDecoderLM(
    sequence_vocabulary_size,
    embedding_width=EMBEDDING_WIDTH,
    hidden_width=48,
)
sequence_embedding_before = sequence_model.embedding.weight.detach().clone()
# [3, 5] -> two [3, 3/4] probe tensors using the same preparation as the full dataset.
cat_sources = source_sequences[-3:]
cat_answers = sequence_targets[-3:]
with torch.inference_mode():
    writer_before_logits = sequence_model(cat_sources, cat_answers)
    writer_before = {
        "sat": word_probe(writer_before_logits[0, 0], "sat", sequence_itos),
        "mat": word_probe(writer_before_logits[2, 1], "mat", sequence_itos),
    }
sequence_optimizer = torch.optim.Adam(sequence_model.parameters(), lr=0.008)
assert EncoderDecoderLM.fit_language_model is TrainableLanguageModel.fit_language_model
sequence_started = time.perf_counter()
sequence_losses = []
sequence_learning_history = [{
    "epoch": 0, "train_loss": None, "sat_probability": writer_before["sat"]["probability"],
    "embedding_shift": 0.0,
}]
print("Reader/writer epoch | practice loss | P(sat | source, <bos>) | shared table change")
print(f"0 | not trained | {writer_before['sat']['probability']:.3%} | 0")
for epoch in range(1, 7):
    epoch_loss = sequence_model.fit_language_model(
        sequence_dataloader, criterion, sequence_optimizer, epochs=1,
    )[0]
    sequence_losses.append(epoch_loss)
    with torch.inference_mode():
        epoch_writer = word_probe(sequence_model(cat_sources[:1], cat_answers[:1])[0, 0],
                                  "sat", sequence_itos)
        sequence_shift = (sequence_model.embedding.weight - sequence_embedding_before).norm().item()
    sequence_learning_history.append({
        "epoch": epoch, "train_loss": epoch_loss,
        "sat_probability": epoch_writer["probability"], "embedding_shift": sequence_shift,
    })
    print(f"{epoch} | {epoch_loss:.4f} | {epoch_writer['probability']:.3%} | {sequence_shift:.4f}")
sequence_seconds = time.perf_counter() - sequence_started
sequence_embedding_after = sequence_model.embedding.weight.detach().clone()
sequence_embedding_shifts = (sequence_embedding_after - sequence_embedding_before).norm(dim=1)
for word in ("the", "sleepy", "cat", "sat", "on", "<bos>", "<eos>"):
    index = sequence_itos.index(word)
    print(f"{word} (fixed ID {index}), first three shared features: "
          f"{sequence_embedding_before[index, :3].tolist()} -> "
          f"{sequence_embedding_after[index, :3].tolist()}; "
          f"row change {sequence_embedding_shifts[index].item():.4f}")
assert torch.any(sequence_embedding_shifts > 0)
# EOS is checked as an answer but is never looked up by this untied model.
assert torch.equal(sequence_embedding_after[EOS_ID], sequence_embedding_before[EOS_ID])
print("Encoder and decoder input rows learned together; the unread EOS input row stayed unchanged.")

with torch.inference_mode():
    # [N, ...] -> first at most 4096 rows: bounded diagnostic avoids a huge [N, 3, V] allocation.
    teacher_forced_logits = sequence_model(source_sequences[:4096], sequence_targets[:4096])
    teacher_forced_predictions = teacher_forced_logits.argmax(dim=-1)
    expected_sequence_ids = sequence_targets[:4096, 1:]
    teacher_forced_accuracy = (
        teacher_forced_predictions == expected_sequence_ids
    ).float().mean().item()
    # [probe, 3] -> [probe, 2]: exclude the easy final stop target when checking content.
    content_accuracy = (
        teacher_forced_predictions[:, :-1] == expected_sequence_ids[:, :-1]
    ).float().mean().item()

print(f"Seq2Seq loss: {sequence_losses[0]:.4f} -> {sequence_losses[-1]:.4f}")
print(f"Encoder-decoder: {sequence_seconds:.1f}s (6 epochs)")
print("Teacher-forced accuracy on the first 4096 training windows:", f"{teacher_forced_accuracy:.1%}")
print("Without the predictable stop target:", f"{content_accuracy:.1%}")
print("Sequence logits:", tuple(teacher_forced_logits.shape))
with torch.inference_mode():
    writer_after_logits = sequence_model(cat_sources, cat_answers)
    report_probe("Writer next 'sat' after source 'the sleepy cat' + <bos>",
                 writer_before["sat"], word_probe(writer_after_logits[0, 0], "sat", sequence_itos))
    report_probe("Writer next 'mat' after source 'cat sat on' + correct previous 'the'",
                 writer_before["mat"], word_probe(writer_after_logits[2, 1], "mat", sequence_itos))

assert sequence_losses[-1] < sequence_losses[0]
assert sequence_losses[-1] < math.log(sequence_vocabulary_size)
assert teacher_forced_logits.shape == (min(4096, len(windows)), 3, sequence_vocabulary_size)
print(f"Introductory training total: {cbow_seconds + skipgram_seconds + sequence_seconds:.1f}s")

The before/after row table includes source words, writer-input words, and
the start marker. Those actual values make embedding learning visible.
`<eos>` is different: here it is only a checked answer, never a fetched
input, so its untied input row remains unchanged even while its output
score learns. Repeated epochs do not change that role.

The probability and accuracy above use correct previous words. They are
**teacher-forced training diagnostics**, not a held-out quality claim or
proof that the writer can generate the expected continuation by itself.
Section 6.4 removes that answer key; 6.5 onward keeps the original
independent held-out comparison.


### 6.4 - Remove the answer key

Teacher-forced accuracy is not free-running generation. Feed each greedy guess
back into the decoder, then inspect the actual continuation. Reverse the source
to check that the GRU's final state notices order; that alone does not prove a
useful understanding of it.


In [ ]:
# %% 6.4 - Generate without correct previous answers
def greedy_decode(model, source_ids, bos_id, eos_id, max_steps):
    model.eval()
    current_ids = torch.full(
        (len(source_ids), 1),
        bos_id,
        dtype=torch.long,
    )
    generated = []

    with torch.inference_mode():
        hidden = model.encode(source_ids)
        for _ in range(max_steps):
            logits, hidden = model.decode(current_ids, hidden)
            # [B, 1, V] -> [B, V] -> [B]: choose only the latest step's best word.
            next_ids = logits[:, -1].argmax(dim=-1)
            generated.append(next_ids)
            # [B] -> [B, 1]: a new one-token time axis lets the GRU read its own guess.
            current_ids = next_ids.unsqueeze(1)
            if bool((next_ids == eos_id).all()):
                break

    # List of T [B] columns -> [B, T]: keep generated words in time order.
    return torch.stack(generated, dim=1)


generated_ids = greedy_decode(
    sequence_model,
    cat_sources[:1],
    BOS_ID,
    EOS_ID,
    max_steps=sequence_targets.shape[1] - 1,
)

with torch.inference_mode():
    ordered_hidden = sequence_model.encode(cat_sources[:1])
    reversed_hidden = sequence_model.encode(cat_sources[:1].flip(dims=[1]))

expected_words = [
    sequence_itos[index]
    for index in cat_answers[0, 1:].tolist()
]
generated_words = [
    sequence_itos[index]
    for index in generated_ids[0].tolist()
]
hidden_difference = (ordered_hidden - reversed_hidden).abs().max().item()

print("Expected continuation:", expected_words)
print("Greedy continuation:", generated_words)
print(f"Hidden-state difference after reversal: {hidden_difference:.6f}")

assert generated_ids.shape[0] == 1
assert generated_ids.shape[1] >= 1
assert hidden_difference > 0


The reader passes only its final state. A detail lost from that state cannot be
reopened by this writer. [03D](03D-recurrent-encoder-decoder-to-attention.ipynb)
adds access to saved source states; the
[Transformer track](../../04-transformers/README.md) later builds contextual
representations. Neither mechanism is implemented by this preview.


#### Independent comparison - keep the three original training experiments

### 6.5 - Load the comparison split

Start fresh models on **8,000 stories**, split **80/20 by whole story before
vocabulary construction or windows**. Use 48-number rows with a common start,
40,000 training windows, 4,000 held-out windows, and 16 epochs.

This extension has its **own** data split and does not continue the core model.
The core's 200 reserved stories evaluate only that core model; do not treat them
as an untouched test after training these new models on their different split.

**Predict:** does falling practice loss guarantee better held-out predictions
or useful geometry among the named probe words?


These are **new** CBOW, Skip-gram, and encoder-decoder instances, not continuation
of section 3.3 or 6.3. Their common starting rows make the larger experiment
controlled. For this comparison the writer predicts two continuation words
without an easy final EOS target; the smaller section-6 run above includes EOS.


In [ ]:
# %% 6.5 - Load the independent comparison split
from pathlib import Path

import real_text_embeddings
from real_text_embeddings import (
    animate_word_maps,
    blank_shortlist,
    load_real_text,
    plot_losses_and_geometry,
    plot_similarity_heatmaps,
    plot_word_maps,
    project_snapshots,
    report_neighbours,
    shortlist_summary,
    train_comparison,
)
from real_text_embeddings import best_versus_final

real_corpus = load_real_text()


### 6.6 - Same starting rows, different jobs

| Model | Input from each window | Prediction |
|---|---|---|
| CBOW | Four neighbours | Center |
| Skip-gram | Center | One neighbour per pair |
| Reader/writer | First three tokens | Last two tokens in order |

Each gets its own aligned pairs and loader, but the same inherited optimizer
routine. This comparison omits the writer's easy stop target. Epoch zero is the
actual shared initialization.

Target counts and parameter counts differ, so this is not a fair-speed contest
or a causal ranking of architectures. Monitor each objective's held-out curve;
do not rank unlike tasks by raw loss alone.


In [ ]:
# %% 6.6 - Record all three training histories
REAL_TEXT_EPOCHS = 16
real_results = train_comparison(
    real_corpus, CBOW, SkipGram, EncoderDecoderLM, epochs=REAL_TEXT_EPOCHS,
)


### 6.7 - Compare practice and held-out trajectories

Inspect the actual curves for divergence. The geometry panel compares within-
group and across-group cosine; group labels are diagnostic only and never enter
training. The best held-out checkpoint need not be the final epoch.


In [ ]:
# %% 6.7 - Plot recorded loss and geometry
chart_directory = Path(real_text_embeddings.__file__).resolve().parent / "images"
loss_figure = plot_losses_and_geometry(real_results)
loss_figure.savefig(chart_directory / "03a-real-text-loss-and-similarity.png", dpi=130)
plt.show()
plt.close(loss_figure)


![Measured training and held-out loss for all three objectives, with within-group and across-group cosine similarity below.](images/03a-real-text-loss-and-similarity.png)
*Saved from the measured run; running the cell above regenerates this chart.*

Here's the complaint the tiny corpus hid: **we can get better at the practice questions and
worse at new passages at the same time.** The best held-out epochs were 5 for CBOW, 3 for
Skip-gram, and 4 for the writer. We deliberately show all 16 epochs instead of cropping off
the awkward part. These held-out stories are monitoring data, not an untouched final test set.

Now look at the writer's bottom panel. Its gap shoots up fastest (about **+0.54** by
epoch 2, higher than either other model ever gets), then slides down for the rest of training.
Hang on to that; we'll come back to it.


### 6.8 - Keep the projection camera fixed

One PCA projection and fixed axes cover all epochs and objectives. It is a view
of a 48-dimensional trajectory, not the whole representation.


In [ ]:
# %% 6.8 - Project all histories through one camera
real_probe_words, real_coordinates, real_pca_fraction = project_snapshots(
    real_corpus, real_results,
)
map_figure = plot_word_maps(real_probe_words, real_coordinates, real_results)
map_figure.savefig(chart_directory / "03a-real-text-word-maps.png", dpi=130)
plt.show()
plt.close(map_figure)


![The same sixteen words at initialization and after training for CBOW, Skip-gram, and the encoder-decoder, using one frozen PCA projection.](images/03a-real-text-word-maps.png)
*The colors name our inspection groups. Training never receives those colors.*


### 6.9 - Check full-row similarity too

These heatmaps compare all 48 coordinates with a shared scale. The omitted
diagonal is self-similarity. Inspect both within-group blocks and cross-group
values rather than accepting tidy islands in a two-dimensional map.


In [ ]:
# %% 6.9 - Compare complete stored rows
similarity_figure = plot_similarity_heatmaps(real_corpus, real_results)
similarity_figure.savefig(chart_directory / "03a-real-text-cosine-heatmaps.png", dpi=130)
plt.show()
plt.close(similarity_figure)


![Cosine similarity heatmaps for the common random start and each trained model, computed in all 48 embedding dimensions with a shared scale.](images/03a-real-text-cosine-heatmaps.png)
*This checks the rows directly. It doesn't depend on the PCA camera.*


### 6.10 - Optional recorded playback

Each frame uses a saved epoch, not invented interpolation. The static plots
above remain the evidence when inline playback is unavailable.


In [ ]:
# %% 6.10 - Play the recorded snapshots
real_animation = animate_word_maps(real_probe_words, real_coordinates, real_results)
real_animation_html = real_animation.to_jshtml()
if "get_ipython" in globals():
    from IPython.display import HTML, display

    display(HTML(real_animation_html))
else:
    print(f"Recorded animation rendered ({len(real_animation_html):,} HTML characters).")


### 6.11 - Inspect the neighbours behind the picture

Report nearest probe neighbours and full-vocabulary neighbours before and after
training. A small named probe set is a sanity check, not overall semantic quality.
Compare actual pairs against across-group similarity to detect a general collapse
of all rows into one direction.


In [ ]:
# %% 6.11 - Print actual neighbour changes
report_neighbours(real_corpus, real_results)


**Your turn:** change only `probe_pair`, without retraining. Predict the direction
of its change and compare the result with the across-group baseline.


In [ ]:
# %% 6.11 - Your turn: Inspect another pair
probe_pair = ("mom", "dad")  # CHANGE THIS: try ("mom", "green"), without retraining
for objective, record in real_results.items():
    pair_ids = [real_corpus["word_ids"][word] for word in probe_pair]
    pair_scores = []
    for rows in (record["snapshots"][0], record["snapshots"][-1]):
        pair_vectors = torch.nn.functional.normalize(rows[pair_ids], dim=1)
        pair_scores.append((pair_vectors[0] @ pair_vectors[1]).item())
    print(f"{objective}: {' / '.join(probe_pair)} cosine "
          f"{pair_scores[0]:+.3f} -> {pair_scores[1]:+.3f}; "
          f"final across-group mean {record['across'][-1]:+.3f}")


### 6.12 - A missing word can have several plausible answers

Use CBOW's best held-out checkpoint. Inspect the full shortlist, correct-answer
frequency, confidence, and row similarity rather than showing only a friendly
top guess. Similar usage does not imply synonyms or interchangeability.


In [ ]:
# %% 6.12 - Check held-out answer shortlists
cbow_best = real_results["CBOW"]["best_model"]
for left, right in [("was so", "to have"), ("a little", "named tom"),
                    ("played with", "toys and"), ("they went", "the park")]:
    menu = blank_shortlist(cbow_best, real_corpus, left, right)
    print(f"{left} ___ {right} -> " + ", ".join(f"{word} {chance:.0%}" for word, chance in menu))

shortlist = shortlist_summary(cbow_best, real_corpus)
print(f"\nAll {shortlist['blanks']:,} held-out blanks with a known answer:")
print(f"  right answer is the number-one guess:   {shortlist['first_right']:.1%}")
print(f"  right answer is somewhere in the top 5: {shortlist['shortlist_right']:.1%}")
print(f"  average share given to the first guess: {shortlist['first_share']:.1%}")
print(f"  how alike the 5 shortlisted words' rows are: {shortlist['shortlist_similarity']:+.3f}")
print(f"  how alike 5 random words' rows are:          {shortlist['random_similarity']:+.3f}")

assert shortlist["shortlist_right"] > shortlist["first_right"]
assert shortlist["shortlist_similarity"] > shortlist["random_similarity"]


### 6.13 - Compare the writer's best checkpoint with its last

Does a weaker final geometry plot mean the writer never learned useful rows?
Read its recorded best-versus-final values. A fading gap after held-out loss
stops improving is evidence about this run, not proof that extra recurrent
parameters caused it; objectives, target counts, and computation also differ.


In [ ]:
# %% 6.13 - Check best versus final without retraining
writer_check = best_versus_final(real_results)
writer_record = writer_check["Encoder-decoder"]
if writer_record["final_gap"] < writer_record["best_gap"]:
    print("The writer's group separation faded after its best held-out checkpoint.")
else:
    print("This run did not show fading group separation; judge the recorded losses too.")


The message branches on the recorded outcome. Extra-seed runs below are off by
default and cost another full comparison each; they provide no evidence unless run.


In [ ]:
# %% 6.13 - Optional extra-seed robustness check
RUN_SEED_CHECK = False
if RUN_SEED_CHECK:
    for check_seed in (5, 123):
        print(f"--- seed {check_seed}")
        best_versus_final(train_comparison(
            real_corpus, CBOW, SkipGram, EncoderDecoderLM,
            epochs=REAL_TEXT_EPOCHS, seed=check_seed, verbose=False,
        ))
else:
    print("Skipped. Set RUN_SEED_CHECK = True to retrain with seeds 5 and 123.")
print(f"Whole default lesson: {time.perf_counter() - notebook_started:.1f}s "
      "(cache verified, one CPU thread; first download is extra).")


## Section 6 checkpoint - What did the reader/writer add?

**Built and measured:** fused lookup equality, window playback, other objectives,
greedy generation, and the larger held-out comparison.
**Explained:** static stored rows versus recurrent working states.
**Left for later:** negative sampling, weight tying, variable-length batches,
beam search, and attention.

None of these extensions changes the core conclusion: training can improve
stored features, but it cannot recover order discarded before prediction.


The reader/writer learned ordered recurrent states and updated its shared input
profiles through both reading and writing. Its final-state bottleneck remains:
the writer cannot revisit source details absent from the reader's final note.
The [03D attention lesson](03D-recurrent-encoder-decoder-to-attention.ipynb)
addresses that access problem. No original comparison experiment was removed.
